# Import Packages

In [1]:
import pandas as pd
import numpy as np
import mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

import warnings
warnings.filterwarnings('ignore')

# Import Data

In [2]:
df = pd.read_csv('Airline_clean.csv')
df.head(5)

,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Departure Date,Airport Code,Pilot Name,Flight Status,passenger_count,is_international,age_group
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,28/06/2022,CXF,Fransisco Hazeldine,On Time,1,1,Middle Aged (45-64)
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,26/12/2022,YCO,Marla Parsonage,On Time,1,1,Middle Aged (45-64)
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,18/01/2022,GNB,Rhonda Amber,On Time,1,1,Senior (65+)
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,16/09/2022,YND,Kacie Commucci,Delayed,1,1,Senior (65+)
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,25/02/2022,SEE,Ebonee Tree,On Time,1,1,Young Adult (18-24)


# Data Setup

In [3]:
#Select relevant columns

cols = ['Gender', 'age_group', 'Nationality', 'Airport Continent', 'Flight Status']
df_arm = df[cols].copy()

In [4]:
#Convert to transaction format

df_arm = df[cols].astype(str)

# Convert to transactions
transactions = df_arm.apply(lambda row: [f"{col}={val}" for col, val in row.items()], axis=1).tolist()

# Preview first transaction
transactions[0]

['Gender=Female',
 'age_group=Middle Aged (45-64)',
 'Nationality=Japan',
 'Airport Continent=NAM',
 'Flight Status=On Time']

In [5]:
# Data transformation

#TransactionEncoder() function can only handle string type
df_arm = df[cols].astype(str)

#TransactionEncoder() was designed to convert lists to array
list_transactions = df_arm.values.tolist()

#Convert the list to one-hot encoded boolean numpy array
#Apriori function allows boolean data type only, such as 1 and 0, or FALSE and TRUE
te = TransactionEncoder()
array_te = te.fit(list_transactions).transform(list_transactions)

#Check the array
array_te

#Check the columns
te.columns_

#Apriori function can handle dataframe only, convert the array to a dataframe
arm_df = pd.DataFrame(array_te, columns=te.columns_)
arm_df

,AF,AS,Adult (25-44),Afghanistan,Aland Islands,Albania,Algeria,American Samoa,Andorra,Angola,...,Uzbekistan,Vanuatu,Venezuela,Vietnam,Wallis and Futuna,Western Sahara,Yemen,Young Adult (18-24),Zambia,Zimbabwe
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97688,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
97689,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
97690,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
97691,True,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# Min Support & Finding Frequent Itemsets

In [6]:
#Find frequent itemsets
frequent_itemsets = apriori(arm_df, min_support=0.05, use_colnames=True)

#Add length column
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

#Check total found
print(f"Number of frequent itemsets found: {len(frequent_itemsets)}")

#Preview top itemsets
frequent_itemsets.sort_values('support', ascending=False).head(10)

Number of frequent itemsets found: 84


,support,itemsets,length
10,0.503025,(Male),1
8,0.496975,(Female),1
3,0.334118,(Cancelled),1
14,0.333105,(On Time),1
6,0.332777,(Delayed),1
12,0.327680,(NAM),1
18,0.287994,(Senior (65+)),1
11,0.224059,(Middle Aged (45-64)),1
2,0.222298,(Adult (25-44)),1
1,0.190771,(AS),1


# Min Confidence & Generating Association Rules

In [7]:
# Generate association rules with minimum confidence of 0.3
rules_con = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)

# Check total rules generated
print(f"Total rules generated: {len(rules_con)}")
rules_con.head()

Total rules generated: 85


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(AF),(Female),0.112905,0.496975,0.056811,0.503173,1.012471,1.0,0.000700,1.012475,0.013885,0.102719,0.012321,0.308743
1,(AF),(Male),0.112905,0.503025,0.056094,0.496827,0.987679,1.0,-0.000700,0.987682,-0.013868,0.100197,-0.012471,0.304170
2,(AS),(Cancelled),0.190771,0.334118,0.063822,0.334550,1.001291,1.0,0.000082,1.000648,0.001594,0.138423,0.000648,0.262783
3,(AS),(Delayed),0.190771,0.332777,0.063055,0.330525,0.993233,1.0,-0.000430,0.996636,-0.008349,0.136928,-0.003375,0.260003
4,(AS),(Female),0.190771,0.496975,0.093845,0.491925,0.989837,1.0,-0.000963,0.990059,-0.012528,0.158014,-0.010040,0.340379


# Lift Threshold

In [8]:
#Keep only rules where Flight Status is the outcome (right-hand side)
#Filter rules where Flight Status is on the right-hand side only
flight_status_values = ['Cancelled', 'On Time', 'Delayed']

flight_status_rules = rules_con[
    rules_con['consequents'].apply(
        lambda x: any(item in flight_status_values for item in x)
    )
]

#Rank the rules: highest lift first, then highest confidence
flight_status_rules = flight_status_rules.sort_values(
    by=['lift', 'confidence'],
    ascending=False
)

#Show only the important columns
result_arm = flight_status_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

#Results
print(f"Total Flight Status rules found: {len(result_arm)}")
result_arm

Total Flight Status rules found: 33


,antecedents,consequents,support,confidence,lift
62,(Middle Aged (45-64)),(On Time),0.075195,0.335602,1.007498
40,(Senior (65+)),(Delayed),0.096435,0.334850,1.006228
74,"(Female, NAM)",(Delayed),0.054067,0.334600,1.005478
6,(AS),(On Time),0.063894,0.334925,1.005465
32,(China),(On Time),0.062215,0.334858,1.005262
15,(Female),(Cancelled),0.166859,0.335750,1.004884
22,(Senior (65+)),(Cancelled),0.096670,0.335667,1.004637
70,"(Male, NAM)",(Cancelled),0.055746,0.335634,1.004538
39,(NAM),(Delayed),0.109414,0.333906,1.003392
57,(Male),(On Time),0.168057,0.334093,1.002967


# Result

In [9]:
#Display the results
print(f"Total Flight Status rules found: {len(result_arm)}")
result_arm

Total Flight Status rules found: 33


,antecedents,consequents,support,confidence,lift
62,(Middle Aged (45-64)),(On Time),0.075195,0.335602,1.007498
40,(Senior (65+)),(Delayed),0.096435,0.334850,1.006228
74,"(Female, NAM)",(Delayed),0.054067,0.334600,1.005478
6,(AS),(On Time),0.063894,0.334925,1.005465
32,(China),(On Time),0.062215,0.334858,1.005262
15,(Female),(Cancelled),0.166859,0.335750,1.004884
22,(Senior (65+)),(Cancelled),0.096670,0.335667,1.004637
70,"(Male, NAM)",(Cancelled),0.055746,0.335634,1.004538
39,(NAM),(Delayed),0.109414,0.333906,1.003392
57,(Male),(On Time),0.168057,0.334093,1.002967
